In [ ]:
import pyarrow.parquet as pq
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from python.extraction import extract_filenames
from python.extraction import extract_ranges
from python.extraction import extract_profiles
from python.helpfn_for_prediction import set_index_if_exists
from python.helpfn_for_prediction import adjust_genomic_positions_for_bed
from python.helpfn_for_prediction import move_metadata_columns_to_ranges

import os
import pickle

In [ ]:
# directories
working_dir = "/Users/natsudanav/Desktop/PRIMEloci/evaluation/model_evaluation"
model_dir = "/Users/natsudanav/Documents/data_PRIMEloci_dev/model_PRIMEloci/PRIMEloci_GM12878_model_1.0_trvl.sav"
profile_dir = "/Users/natsudanav/Documents/data_PRIMEloci_dev/GM12878_wt10M_profiles_te/profiles_subtnorm"

os.chdir(working_dir)

In [ ]:
# Load the trained model
with open(model_dir, 'rb') as file:
    model = pickle.load(file)

In [ ]:
#Test data
profile_file_ls = extract_filenames(profile_dir)

pos_te_list = [element for element in profile_file_ls if "_pos_" in element]
neg_te_list = [element for element in profile_file_ls if "_neg_" in element]

In [ ]:
all_profiles = []

for profile_filename in profile_file_ls:
    filenames_without_extensions = os.path.splitext(profile_filename)[0]
    input_profiles_subtnorm_path = os.path.join(profile_dir, profile_filename)
    subtnorm_df = pd.read_csv(input_profiles_subtnorm_path, header=0, index_col=None)
    subtnorm_df = set_index_if_exists(subtnorm_df)
    subtnorm_np = extract_profiles(subtnorm_df)
    all_profiles.append(subtnorm_np)

if all_profiles:
    concatenated_profiles = np.concatenate(all_profiles, axis=0)
    print("Final concatenated array shape:", concatenated_profiles.shape)
else:
    print("No profiles were processed.")

In [ ]:
pos_profiles = []

for profile_filename in pos_te_list:
    filenames_without_extensions = os.path.splitext(profile_filename)[0]
    input_profiles_subtnorm_path = os.path.join(profile_dir, profile_filename)
    subtnorm_df = pd.read_csv(input_profiles_subtnorm_path, header=0, index_col=None)
    subtnorm_df = set_index_if_exists(subtnorm_df)
    subtnorm_np = extract_profiles(subtnorm_df)
    pos_profiles.append(subtnorm_np)

if pos_profiles:
    concatenated_profiles = np.concatenate(pos_profiles, axis=0)
    print("Final concatenated array shape:", concatenated_profiles.shape)
else:
    print("No profiles were processed.")

In [ ]:
neg_profiles = []

for profile_filename in neg_te_list:
    filenames_without_extensions = os.path.splitext(profile_filename)[0]
    input_profiles_subtnorm_path = os.path.join(profile_dir, profile_filename)
    subtnorm_df = pd.read_csv(input_profiles_subtnorm_path, header=0, index_col=None)
    subtnorm_df = set_index_if_exists(subtnorm_df)
    subtnorm_np = extract_profiles(subtnorm_df)
    neg_profiles.append(subtnorm_np)

if neg_profiles:
    concatenated_profiles = np.concatenate(neg_profiles, axis=0)
    print("Final concatenated array shape:", concatenated_profiles.shape)
else:
    print("No profiles were processed.")

In [ ]:
# Use SHAP for explanations

In [ ]:
# Ensure the model is a valid LightGBM object
print(type(model))  # Confirm the model is an LGBMClassifier

# SHAP TreeExplainer for LightGBM
explainer = shap.TreeExplainer(model)

In [ ]:
if isinstance(all_profiles, list):
    all_profiles_np = np.vstack(all_profiles)
    
shap_values = explainer.shap_values(all_profiles_np)

In [ ]:
# Plot summary for all features
shap.summary_plot(shap_values, all_profiles_np)

# SHAP values for positive (class 1) and negative (class 0)
shap_values_pos = shap_values[:, :, 1]  # SHAP values for positive class
shap_values_neg = shap_values[:, :, 0]  # SHAP values for negative class

# Feature importance for positive predictions
print("Top features contributing to positive predictions:")
print(np.mean(shap_values_pos.values, axis=0))

# Feature importance for negative predictions
print("Top features contributing to negative predictions:")
print(np.mean(shap_values_neg.values, axis=0))

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.utils import shuffle

# Assuming you have a trained LightGBM model and test data
# test_X: test input features (Pandas DataFrame)
# test_y: test labels (Pandas Series or array)
# model: trained LightGBM model

# Function to calculate Permutation Feature Importance
def permutation_feature_importance(model, X, y, metric=roc_auc_score, n_repeats=5):
    baseline_score = metric(y, model.predict_proba(X)[:, 1])  # Base performance
    importance = pd.DataFrame(index=X.columns, columns=range(n_repeats))

    for col in X.columns:
        for i in range(n_repeats):
            X_permuted = X.copy()
            X_permuted[col] = shuffle(X[col])  # Shuffle feature values
            permuted_score = metric(y, model.predict_proba(X_permuted)[:, 1])
            importance.loc[col, i] = baseline_score - permuted_score  # Drop in performance

    return importance.mean(axis=1).sort_values(ascending=False), importance

# Split data by predictions and true labels
pred_probs = model.predict_proba(test_X)[:, 1]
pos_indices_pred = pred_probs >= 0.5
neg_indices_pred = pred_probs < 0.5
pos_indices_true = test_y == 1
neg_indices_true = test_y == 0

# Permutation Feature Importance
importance_pos_pred, importance_pos_pred_repeats = permutation_feature_importance(model, test_X[pos_indices_pred], test_y[pos_indices_pred])
importance_neg_pred, importance_neg_pred_repeats = permutation_feature_importance(model, test_X[neg_indices_pred], test_y[neg_indices_pred])

# SHAP values
explainer = shap.Explainer(model, test_X)
shap_values = explainer(test_X)

# Visualization of SHAP and Permutation Feature Importance

# Combine SHAP and PFI
def plot_combined_importance(importance, shap_values, subset_indices, title):
    shap_summary = shap_values[subset_indices]
    shap_importance = np.abs(shap_summary.values).mean(axis=0)
    shap_importance_df = pd.Series(shap_importance, index=test_X.columns).sort_values(ascending=False)

    # Combine with PFI
    combined = pd.DataFrame({
        "SHAP Importance": shap_importance_df,
        "PFI Importance": importance.reindex(shap_importance_df.index)
    }).dropna()

    # Plot
    combined.plot(kind="bar", figsize=(12, 6), alpha=0.8)
    plt.title(f"Combined SHAP and PFI: {title}")
    plt.ylabel("Importance")
    plt.xlabel("Features")
    plt.tight_layout()
    plt.savefig(f"combined_importance_{title.replace(' ', '_').lower()}.png")
    plt.show()

# Plot for predicted positives
plot_combined_importance(importance_pos_pred, shap_values, pos_indices_pred, "Predicted Positive")

# Plot for predicted negatives
plot_combined_importance(importance_neg_pred, shap_values, neg_indices_pred, "Predicted Negative")

# Plot for true positives
plot_combined_importance(importance_pos_pred, shap_values, pos_indices_true, "True Positive")

# Plot for true negatives
plot_combined_importance(importance_neg_pred, shap_values, neg_indices_true, "True Negative")
